In [ ]:
!pip install fastapi uvicorn transformers torch sentencepiece
!pip install pyngrok

In [ ]:
import os
import time
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import uvicorn
import threading
from pyngrok import ngrok

In [ ]:
class EmbedRequest(BaseModel):
    texts: List[str]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-large")
model = AutoModel.from_pretrained("intfloat/multilingual-e5-large").to(device)
model.eval()

def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(
        ~attention_mask[..., None].bool(), 0.0
    )
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

### Routes

In [ ]:
app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/embed")
def embed(request: EmbedRequest):

    all_embeddings = []

    batch_size = 32  # safe for T4

    for i in range(0, len(request.texts), batch_size):

        batch = request.texts[i:i+batch_size]
        batch = ["passage: " + t for t in batch]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoded)

        embeddings = average_pool(
            outputs.last_hidden_state,
            encoded["attention_mask"]
        )

        embeddings = F.normalize(embeddings, p=2, dim=1)

        all_embeddings.extend(embeddings.cpu().tolist())

    return {"embeddings": all_embeddings}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()

---

## Ngrok Tunnel

In [ ]:
os.environ["NGROK_AUTHTOKEN"] = "39I7VC4uhq8ygeXGxH7yU0TR8yU_2gNooCY8CdU5fP89JH5FT"
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

ngrok.kill()
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

In [ ]:
!pkill -f ngrok
!pkill -f uvicorn

In [ ]:
def keep_alive():
    while True:
        print("Heartbeat...")
        time.sleep(60)

keep_alive()

Heartbeat...
Heartbeat...
Heartbeat...


KeyboardInterrupt: 


---



## Cloudflared Tunnel

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.2.0) ...
Setting up cloudflared (2026.2.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!cloudflared tunnel --url http://localhost:8000

2026-02-22T02:39:36Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-02-22T02:39:36Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-02-22T02:39:41Z INF +--------------------------------------------------------------------------------------------+
2026-02-22T02:39:41Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-02-22T02:39:41Z INF |  https://furnished-villages-trusts-memorial.trycloudfl

## Tests

---

In [ ]:
import requests
requests.get("http://localhost:8000/health").json()

INFO:     127.0.0.1:55766 - "GET /health HTTP/1.1" 200 OK


{'status': 'ok'}

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

r = requests.get(
    "https://inadvertently-measureless-ester.ngrok-free.dev/health",
    headers=headers
)

print(r.status_code)

404


In [ ]:
# requests.post(
#     "https://inadvertently-measureless-ester.ngrok-free.dev/embed",
#     json={"texts": ["hello world"]}
# ).json()